# Robin Marte – Week 5 Assignment

Questions:
1. What is the northernmost airport in the United States?
2. What is the easternmost airport in the United States?
3. On February 12th, 2013, which New York area airport had the windiest weather?

In [2]:
import pandas as pd
import numpy as np

In [3]:
airports_url = "https://raw.githubusercontent.com/tidyverse/nycflights13/main/data-raw/airports.csv"
weather_url  = "https://raw.githubusercontent.com/tidyverse/nycflights13/main/data-raw/weather.csv"

airports = pd.read_csv(airports_url)
weather = pd.read_csv(weather_url)

airports.head()

,faa,name,lat,lon,alt,tz,dst,tzone
0,04G,Lansdowne Airport,41.130472,-80.619583,1044,-5,A,America/New_York
1,06A,Moton Field Municipal Airport,32.460572,-85.680028,264,-6,A,America/Chicago
2,06C,Schaumburg Regional,41.989341,-88.101243,801,-6,A,America/Chicago
3,06N,Randall Airport,41.431912,-74.391561,523,-5,A,America/New_York
4,09J,Jekyll Island Airport,31.074472,-81.427778,11,-5,A,America/New_York


In [4]:
weather.head()

,origin,year,month,day,hour,temp,dewp,humid,wind_dir,wind_speed,wind_gust,precip,pressure,visib,time_hour
0,EWR,2013,1,1,1,39.02,26.06,59.37,270.0,10.35702,NaN,0.0,1012.0,10.0,2013-01-01T06:00:00Z
1,EWR,2013,1,1,2,39.02,26.96,61.63,250.0,8.05546,NaN,0.0,1012.3,10.0,2013-01-01T07:00:00Z
2,EWR,2013,1,1,3,39.02,28.04,64.43,240.0,11.50780,NaN,0.0,1012.5,10.0,2013-01-01T08:00:00Z
3,EWR,2013,1,1,4,39.92,28.04,62.21,250.0,12.65858,NaN,0.0,1012.2,10.0,2013-01-01T09:00:00Z
4,EWR,2013,1,1,5,39.02,28.04,64.43,260.0,12.65858,NaN,0.0,1011.9,10.0,2013-01-01T10:00:00Z


The airports.csv file contains airports with timezones listed, and most U.S. airports have a timezone that starts with "America/".

I’m going to filter to airports where timezone starts with "America/"

In [5]:
us_airports = airports[airports["tzone"].astype(str).str.startswith("America/")].copy()

us_airports.shape, airports.shape

((1435, 8), (1458, 8))

## Question 1: Northernmost Airport

To find the northernmost airport, I sort the airports by latitude in descending order. A higher latitude means the airport is farther north.

The top results give me a short list of candidates.

In [6]:
north_candidates = us_airports.sort_values("lat", ascending=False).head(10)
north_candidates[["faa", "name", "lat", "lon", "tzone"]]

,faa,name,lat,lon,tzone
230,BRW,Wiley Post Will Rogers Mem,71.285446,-156.766003,America/Anchorage
110,AIN,Wainwright Airport,70.638056,-159.994722,America/Anchorage
708,K03,Wainwright As,70.613378,-159.860350,America/Anchorage
152,ATK,Atqasuk Edward Burnell Sr Memorial Airport,70.467300,-157.436000,America/Anchorage
1363,UUK,Ugnu-Kuparuk Airport,70.330833,-149.597500,America/Anchorage
982,NUI,Nuiqsut Airport,70.210000,-151.005556,America/Anchorage
1197,SCC,Deadhorse,70.194750,-148.465167,America/Anchorage
232,BTI,Barter Island Lrrs,70.133989,-143.581867,America/Anchorage
1084,PIZ,Point Lay Lrrs,69.732875,-163.005342,America/Anchorage
824,LUR,Cape Lisburne Lrrs,68.875133,-166.110022,America/Anchorage


### Verifying the Result

The airport with the highest latitude in the filtered dataset is:

BRW – Wiley Post Will Rogers Memorial Airport in Utqiagvik, Alaska.

To confirm this makes sense, I checked external sources. Utqiagvik is commonly recognized as the northernmost city in the United States, and its airport is frequently listed as the northernmost public airport.

## Question 2: Easternmost Airport

To determine the easternmost airport, I sort the airports by longitude.

Because Alaska extends far west and crosses the 180° longitude area, some U.S. airports actually have large positive longitude values. Instead of assuming east means the lowest longitude, I sort by the highest longitude value in the dataset.

In [7]:
east_candidates = us_airports.sort_values("lon", ascending=False).head(10)
east_candidates[["faa", "name", "lat", "lon", "tzone"]]

,faa,name,lat,lon,tzone
1290,SYA,Eareckson As,52.712275,174.113620,America/Anchorage
444,EPM,Eastport Municipal Airport,44.910111,-67.012694,America/New_York
624,HUL,Houlton Intl,46.123083,-67.792056,America/New_York
259,CAR,Caribou Muni,46.871500,-68.017917,America/New_York
1101,PQI,Northern Maine Rgnl At Presque Isle,46.688958,-68.044797,America/New_York
1398,WFK,Northern Aroostook Regional Airport,47.285556,-68.312778,America/New_York
192,BHB,Hancock County - Bar Harbor,44.449769,-68.361565,America/New_York
856,ME5,Banks Airport,44.165389,-68.428167,America/New_York
894,MLT,Millinocket Muni,45.647836,-68.685561,America/New_York
191,BGR,Bangor Intl,44.807444,-68.828139,America/New_York


### Verifying the Result

The airport with the highest longitude value in the dataset is:

SYA – Eareckson Air Station in Shemya, Alaska.

I checked this location externally and confirmed that Shemya Island is in the Aleutian Islands and is often referenced as the easternmost U.S. location by longitude measurement.

## Question 3: Windiest NYC Airport on February 12, 2013

For this question, I'll filter the weather dataset to February 12th, 2013

The NYC airports in this dataset are:
- EWR  
- JFK  
- LGA  

To determine which airport is the windiest, I'll compare wind speed values for that day.

In [8]:
w = weather[(weather["year"] == 2013) &
            (weather["month"] == 2) &
            (weather["day"] == 12)].copy()

w[["origin", "time_hour", "wind_speed", "wind_gust"]].head()

,origin,time_hour,wind_speed,wind_gust
1006,EWR,2013-02-12T05:00:00Z,6.90468,NaN
1007,EWR,2013-02-12T06:00:00Z,9.20624,NaN
1008,EWR,2013-02-12T07:00:00Z,20.71404,25.31716
1009,EWR,2013-02-12T08:00:00Z,1048.36058,NaN
1010,EWR,2013-02-12T09:00:00Z,12.65858,NaN


### Checking for Unusual Values

When I review the wind speed data, I notice one extremely large value that does not seem realistic. This suggests a possible data error or outlier.

To prevent that single value from affecting the result, I replace wind speed values above 200 with NaN for this analysis.

This keeps the comparison based on more realistic weather measurements.

In [9]:
# Identifying maximum before cleaning:
w["wind_speed"].max()

np.float64(1048.36058)

In [10]:
# Removing unrealistic outliers:
w_clean = w.copy()
w_clean.loc[w_clean["wind_speed"] > 200, "wind_speed"] = np.nan

# Comparing maximum wind speed by airport:
w_clean.groupby("origin")["wind_speed"].max()

origin
EWR    21.86482
JFK    20.71404
LGA    23.01560
Name: wind_speed, dtype: float64

### Final Result

After removing the outlier and comparing maximum wind speeds for each airport, LGA records the highest wind speed on February 12, 2013.